# Pipeline de limpieza de las bases de mora

Este notebook prepara `train`, `validation` y `test` con un solo criterio reproducible.

**Principio clave:** las medianas y modas de imputación se calculan únicamente con `train`, para evitar fuga de información.

### Pasos
1. Carga de los tres CSV.
2. Normalización de nombres y textos.
3. Conversión controlada de fechas y variables numéricas.
4. Eliminación de duplicados exactos.
5. Validación de coherencia entre `atraso` y `mora`.
6. Aprendizaje de medianas y modas solamente en `train`.
7. Aplicación de los mismos parámetros a `train`, `validation` y `test`.
8. Conservación de las 35 columnas originales.
9. Exportación de los tres CSV limpios.

**Importante para modelado:** `mora` es la etiqueta y `atraso` está directamente relacionado con esa etiqueta. No conviene usar estas columnas como variables predictoras si el objetivo es predecir `mora`.

In [ ]:
# Importamos pandas para trabajar con los archivos CSV.
import pandas as pd
# Importamos Path para construir rutas de forma segura.
from pathlib import Path
# Mostramos la versión de pandas para facilitar la reproducibilidad.
print("pandas:", pd.__version__)

In [ ]:
# En Google Colab importamos el selector de archivos.
from google.colab import files
# Ejecutamos esta celda y seleccionamos los tres CSV originales.
archivos_subidos = files.upload()
# Mostramos los nombres que llegaron al entorno.
print("Archivos cargados:", list(archivos_subidos.keys()))

In [ ]:
# Definimos la carpeta donde Colab deja los archivos subidos.
BASE = Path("/content")
# Definimos el archivo original de train.
RUTA_TRAIN = BASE / "4_data_train-mora.csv"
# Definimos el archivo original de test.
RUTA_TEST = BASE / "5_data_test-mora.csv"
# Definimos el archivo original de validation.
RUTA_VAL = BASE / "6_data_validation-mora.csv"
# Definimos los nombres de salida.
OUT_TRAIN = BASE / "train_mora_limpio.csv"
# Definimos la salida de validation.
OUT_VAL = BASE / "validation_mora_limpio.csv"
# Definimos la salida de test.
OUT_TEST = BASE / "test_mora_limpio.csv"

# Listamos las columnas numéricas que deben convertirse a número.
COLUMNAS_NUMERICAS = [
    "atraso","edad","dias_lab","exp_sf","ingreso","linea_sf","deuda_sf","score",
    "mora_max_3m","mora_max_6m","mora_max_12m","atrasos_6m_12m","atrasos_12m","dti",
    "cuota_ingreso","saldo_promedio_3m","variacion_depositos_ingresos","num_entidades_deuda",
    "nuevos_creditos_3m_6m","nuevos_creditos_6m","porcentaje_utilizacion_linea",
    "saldo_pendiente_credito","cuotas_restantes","antiguedad_cliente_banco",
    "numero_productos_banco",
]

# Listamos las columnas categóricas que se imputarán con la moda de train.
COLUMNAS_CATEGORICAS = [
    "vivienda","nivel_ahorro","zona","clasif_sbs","nivel_educ",
    "tipo_credito","reestructuraciones_refinanciamientos_previos",
]

# Definimos la columna que contiene la fecha.
COLUMNA_FECHA = "fecha_reporte"
# Definimos la columna objetivo.
COLUMNA_OBJETIVO = "mora"

# Definimos las columnas conceptualmente enteras.
COLUMNAS_ENTERAS = [
    "atraso","edad","dias_lab","exp_sf","score","mora_max_3m","mora_max_6m","mora_max_12m",
    "atrasos_6m_12m","atrasos_12m","num_entidades_deuda","nuevos_creditos_3m_6m",
    "nuevos_creditos_6m","cuotas_restantes","antiguedad_cliente_banco","numero_productos_banco",
]

# Definimos el orden final exacto de las 35 columnas.
ORDEN_COLUMNAS = [
    "ID","fecha_reporte","atraso","vivienda","edad","dias_lab","exp_sf","nivel_ahorro","ingreso",
    "linea_sf","deuda_sf","score","zona","clasif_sbs","nivel_educ","mora_max_3m","mora_max_6m",
    "mora_max_12m","atrasos_6m_12m","atrasos_12m","dti","cuota_ingreso","saldo_promedio_3m",
    "variacion_depositos_ingresos","num_entidades_deuda","nuevos_creditos_3m_6m","nuevos_creditos_6m",
    "porcentaje_utilizacion_linea","saldo_pendiente_credito","cuotas_restantes","tipo_credito",
    "antiguedad_cliente_banco","numero_productos_banco","reestructuraciones_refinanciamientos_previos","mora",
]

In [ ]:
# Normalizamos el nombre de cada columna.
def normalizar_nombre_columna(nombre):
    # Convertimos el nombre a texto.
    nombre = str(nombre)
    # Quitamos espacios exteriores.
    nombre = nombre.strip()
    # Pasamos el nombre a minúsculas.
    nombre = nombre.lower()
    # Convertimos grupos de espacios en un guion bajo.
    nombre = "_".join(nombre.split())
    # Conservamos ID en mayúsculas para mantener el esquema original.
    return "ID" if nombre == "id" else nombre

# Normalizamos cada valor de texto.
def normalizar_texto(valor):
    # Conservamos los nulos como nulos reales.
    if pd.isna(valor):
        return pd.NA
    # Convertimos el valor a texto.
    texto = str(valor)
    # Quitamos espacios de los extremos.
    texto = texto.strip()
    # Unificamos espacios internos repetidos.
    texto = " ".join(texto.split())
    # Definimos textos que equivalen a un dato faltante.
    tokens_nulos = {"", "NA","N/A","NULL","NONE","NAN","SIN DATO","NO DISPONIBLE"}
    # Comprobamos si el valor representa un faltante.
    if texto.upper() in tokens_nulos:
        # Transformamos esos tokens en nulos.
        return pd.NA
    # Devolvemos el texto limpio.
    return texto

In [ ]:
# Cargamos un archivo y ejecutamos la limpieza estructural inicial.
def limpiar_dataframe_base(ruta):
    # Leemos todo como texto para controlar nosotros las conversiones.
    df = pd.read_csv(ruta, dtype="string", encoding="utf-8-sig")
    # Normalizamos los nombres de las columnas.
    df.columns = [normalizar_nombre_columna(c) for c in df.columns]
    # Limpiamos espacios y tokens nulos en todas las columnas.
    for columna in df.columns:
        df[columna] = df[columna].map(normalizar_texto)
    # Convertimos la fecha y dejamos inválidas como nulas.
    df[COLUMNA_FECHA] = pd.to_datetime(df[COLUMNA_FECHA], errors="coerce")
    # Convertimos cada columna numérica a número.
    for columna in COLUMNAS_NUMERICAS:
        df[columna] = pd.to_numeric(df[columna], errors="coerce")
    # Contamos duplicados exactos antes de eliminarlos.
    duplicados = int(df.duplicated().sum())
    # Eliminamos solamente duplicados exactos.
    df = df.drop_duplicates().reset_index(drop=True)
    # Normalizamos la etiqueta de mora.
    df[COLUMNA_OBJETIVO] = df[COLUMNA_OBJETIVO].astype("string").str.upper().str.strip()
    # Devolvemos datos y cantidad de duplicados eliminados.
    return df, duplicados

In [ ]:
# Definimos los rangos esperados de atraso por categoría de mora.
RANGOS_MORA = {
    "AL DÍA": (0,0),
    "MORA DE 1 A 7 DÍAS": (1,7),
    "MORA DE 8 A 30 DÍAS": (8,30),
    "MORA DE 31 A 60 DÍAS": (31,60),
    "MORA DE 61 A 90 DÍAS": (61,90),
    "MORA DE MÁS DE 90 DÍAS": (91,float("inf")),
}

# Validamos la coherencia entre las dos columnas relacionadas.
def validar_mora_atraso(df, nombre):
    # Seleccionamos filas donde ambas variables existen.
    mask = df["atraso"].notna() & df[COLUMNA_OBJETIVO].notna()
    # Inicializamos el contador de inconsistencias.
    inconsistencias = 0
    # Revisamos cada fila válida.
    for _, fila in df.loc[mask, ["atraso",COLUMNA_OBJETIVO]].iterrows():
        # Obtenemos el rango asociado a la categoría.
        rango = RANGOS_MORA.get(str(fila[COLUMNA_OBJETIVO]))
        # Validamos únicamente categorías conocidas.
        if rango is not None:
            # Extraemos límites inferior y superior.
            minimo, maximo = rango
            # Contamos cuando el atraso sale del intervalo esperado.
            if not (minimo <= float(fila["atraso"]) <= maximo):
                inconsistencias += 1
    # Mostramos el resultado.
    print(f"{nombre}: inconsistencias atraso↔mora = {inconsistencias}")
    # Devolvemos el conteo.
    return inconsistencias

In [ ]:
# Cargamos la partición train.
train, duplicados_train = limpiar_dataframe_base(RUTA_TRAIN)
# Cargamos la partición validation.
validation, duplicados_validation = limpiar_dataframe_base(RUTA_VAL)
# Cargamos la partición test.
test, duplicados_test = limpiar_dataframe_base(RUTA_TEST)

# Mostramos el tamaño y los duplicados eliminados.
print("train:", train.shape, "duplicados exactos eliminados:", duplicados_train)
print("validation:", validation.shape, "duplicados exactos eliminados:", duplicados_validation)
print("test:", test.shape, "duplicados exactos eliminados:", duplicados_test)

# Validamos la consistencia de las etiquetas.
validar_mora_atraso(train, "train")
validar_mora_atraso(validation, "validation")
validar_mora_atraso(test, "test")

# Comprobamos que no haya IDs repetidos dentro de cada partición.
print("IDs repetidos train:", train["ID"].duplicated().sum())
print("IDs repetidos validation:", validation["ID"].duplicated().sum())
print("IDs repetidos test:", test["ID"].duplicated().sum())

# Comprobamos que no existan IDs compartidos entre particiones.
print("Cruce train-validation:", len(set(train["ID"].dropna()) & set(validation["ID"].dropna())))
print("Cruce train-test:", len(set(train["ID"].dropna()) & set(test["ID"].dropna())))
print("Cruce validation-test:", len(set(validation["ID"].dropna()) & set(test["ID"].dropna())))

In [ ]:
# Calculamos las medianas únicamente con train.
medianas_train = train[COLUMNAS_NUMERICAS].median(numeric_only=True).to_dict()

# Calculamos las modas únicamente con train.
modas_train = {}
# Recorremos las columnas categóricas.
for columna in COLUMNAS_CATEGORICAS:
    # Eliminamos los nulos antes de obtener la moda.
    serie = train[columna].dropna()
    # Usamos la primera moda disponible o una categoría de respaldo.
    modas_train[columna] = str(serie.mode().iloc[0]) if not serie.empty else "SIN_DATO"

# Mostramos las medianas para dejar trazabilidad del proceso.
print("Medianas aprendidas de train:")
display(pd.Series(medianas_train).round(4))

# Mostramos las modas para dejar trazabilidad del proceso.
print("Modas aprendidas de train:")
display(pd.Series(modas_train))

In [ ]:
# Aplicamos los parámetros aprendidos con train.
def imputar_con_train(df, medianas, modas):
    # Creamos una copia para no alterar el objeto original.
    df = df.copy()
    # Imputamos variables numéricas con medianas de train.
    for columna in COLUMNAS_NUMERICAS:
        df[columna] = df[columna].fillna(medianas[columna])
    # Imputamos variables categóricas con modas de train.
    for columna in COLUMNAS_CATEGORICAS:
        df[columna] = df[columna].fillna(modas[columna])
    # Evitamos identificadores vacíos.
    df["ID"] = df["ID"].fillna("ID_SIN_DATO")
    # Devolvemos el dataframe imputado.
    return df

# Imputamos train usando sus propias estadísticas.
train = imputar_con_train(train, medianas_train, modas_train)
# Imputamos validation usando estadísticas exclusivamente de train.
validation = imputar_con_train(validation, medianas_train, modas_train)
# Imputamos test usando estadísticas exclusivamente de train.
test = imputar_con_train(test, medianas_train, modas_train)

In [ ]:
# Estandarizamos el formato final para exportación.
def preparar_para_exportar(df):
    # Creamos una copia para mantener control del objeto.
    df = df.copy()
    # Convertimos la fecha al formato AAAA-MM-DD.
    df[COLUMNA_FECHA] = df[COLUMNA_FECHA].dt.strftime("%Y-%m-%d")
    # Redondeamos las columnas que representan cantidades enteras.
    for columna in COLUMNAS_ENTERAS:
        df[columna] = df[columna].round().astype("Int64")
    # Convertimos las categorías y el ID a texto.
    for columna in COLUMNAS_CATEGORICAS + ["ID", COLUMNA_OBJETIVO]:
        df[columna] = df[columna].astype("string")
    # Reordenamos para conservar exactamente el esquema original.
    df = df[ORDEN_COLUMNAS]
    # Devolvemos la tabla lista.
    return df

# Aplicamos el formato final a train.
train = preparar_para_exportar(train)
# Aplicamos el formato final a validation.
validation = preparar_para_exportar(validation)
# Aplicamos el formato final a test.
test = preparar_para_exportar(test)

In [ ]:
# Guardamos train limpio en UTF-8.
train.to_csv(OUT_TRAIN, index=False, encoding="utf-8-sig", float_format="%.6f")
# Guardamos validation limpio en UTF-8.
validation.to_csv(OUT_VAL, index=False, encoding="utf-8-sig", float_format="%.6f")
# Guardamos test limpio en UTF-8.
test.to_csv(OUT_TEST, index=False, encoding="utf-8-sig", float_format="%.6f")

# Mostramos las dimensiones finales.
print("train limpio:", train.shape)
print("validation limpio:", validation.shape)
print("test limpio:", test.shape)

In [ ]:
# Contamos los nulos restantes en cada partición.
nulos_train = int(train.isna().sum().sum())
nulos_validation = int(validation.isna().sum().sum())
nulos_test = int(test.isna().sum().sum())

# Mostramos los nulos finales.
print("Nulos finales train:", nulos_train)
print("Nulos finales validation:", nulos_validation)
print("Nulos finales test:", nulos_test)

# Mostramos las primeras filas para una inspección rápida.
display(train.head())

# En Colab se pueden descargar los CSV con estas instrucciones.
# files.download(str(OUT_TRAIN))
# files.download(str(OUT_VAL))
# files.download(str(OUT_TEST))